In [ ]:
# ============================================================
# Cell 1: Imports and Configuration
# ============================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import os
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device selection: CUDA > MPS > CPU
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f"Using device: {device}")

# Hyperparameters
LOOKBACK = 60          # Sequence length (days)
HIDDEN_SIZE = 64       # LSTM hidden units
NUM_LAYERS = 2         # Stacked LSTM layers
DROPOUT = 0.2          # Dropout between LSTM layers
LR = 0.001             # Learning rate
EPOCHS = 300           # Training epochs
BATCH_SIZE = 32        # Batch size
THRESHOLD = 0.005      # Signal threshold (0.5% predicted move)
INITIAL_CAPITAL = 10000.0
TX_FEE = 0.001         # 0.1% transaction fee

# Plot style
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

# Output directory
os.makedirs('results', exist_ok=True)

## 1. Data Loading & Feature Engineering

In [ ]:
# ============================================================
# Cell 2: Load Data and Engineer Features
# ============================================================

# --- Load raw OHLCV data ---
df = pd.read_csv('bitcoin_2023-03-09_2026-03-08.csv', parse_dates=['Start'])
df = df.rename(columns={'Start': 'Date'})
df = df[['Date', 'Open', 'High', 'Low', 'Close', 'Volume']]
df = df.sort_values('Date').reset_index(drop=True)
df = df.set_index('Date')
print(f"Loaded data: {df.shape[0]} rows, {df.index[0].date()} to {df.index[-1].date()}")

# --- Target: log returns ---
df['log_return'] = np.log(df['Close'] / df['Close'].shift(1))

# --- Technical Indicators ---

# Simple Moving Averages
df['SMA_7'] = df['Close'].rolling(window=7).mean()
df['SMA_21'] = df['Close'].rolling(window=21).mean()

# RSI (14-day)
delta = df['Close'].diff()
gain = delta.where(delta > 0, 0.0)
loss = -delta.where(delta < 0, 0.0)
avg_gain = gain.rolling(window=14).mean()
avg_loss = loss.rolling(window=14).mean()
rs = avg_gain / (avg_loss + 1e-10)
df['RSI'] = 100 - (100 / (1 + rs))

# MACD
ema12 = df['Close'].ewm(span=12, adjust=False).mean()
ema26 = df['Close'].ewm(span=26, adjust=False).mean()
df['MACD'] = ema12 - ema26
df['MACD_signal'] = df['MACD'].ewm(span=9, adjust=False).mean()

# Bollinger Band Percentage
sma20 = df['Close'].rolling(window=20).mean()
std20 = df['Close'].rolling(window=20).std()
upper_band = sma20 + 2 * std20
lower_band = sma20 - 2 * std20
df['BB_pct'] = (df['Close'] - lower_band) / (upper_band - lower_band + 1e-10)

# ATR Percentage (14-day)
prev_close = df['Close'].shift(1)
tr = pd.concat([
    df['High'] - df['Low'],
    (df['High'] - prev_close).abs(),
    (df['Low'] - prev_close).abs()
], axis=1).max(axis=1)
atr = tr.rolling(window=14).mean()
df['ATR_pct'] = atr / df['Close']

# Volume Ratio
df['Vol_ratio'] = df['Volume'] / df['Volume'].rolling(window=14).mean()

# Returns
df['Returns_1d'] = df['Close'].pct_change(1)
df['Returns_5d'] = df['Close'].pct_change(5)

# --- Temporal Features (cyclical encoding) ---
dow = df.index.dayofweek
df['dow_sin'] = np.sin(2 * np.pi * dow / 7)
df['dow_cos'] = np.cos(2 * np.pi * dow / 7)

# --- Drop NaN rows ---
df = df.dropna()
print(f"After feature engineering & dropna: {df.shape[0]} rows")

# --- Define feature columns and target ---
feature_cols = [
    'SMA_7', 'SMA_21', 'RSI', 'MACD', 'MACD_signal', 'BB_pct',
    'ATR_pct', 'Vol_ratio', 'Returns_1d', 'Returns_5d',
    'dow_sin', 'dow_cos'
]
target_col = 'log_return'

assert df[feature_cols + [target_col]].isna().sum().sum() == 0, "NaN values remain!"
print(f"Features: {len(feature_cols)}, Target: {target_col}")
df[feature_cols + [target_col]].describe().round(4)

## 2. Train/Test Split, Scaling & Sequence Creation

In [ ]:
# ============================================================
# Cell 3: Chronological Split, Scaling, Sequence Creation
# ============================================================

# --- Chronological 80/20 split (BEFORE sequencing) ---
split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]
print(f"Train: {len(train_df)} rows ({train_df.index[0].date()} to {train_df.index[-1].date()})")
print(f"Test:  {len(test_df)} rows ({test_df.index[0].date()} to {test_df.index[-1].date()})")

# --- Scale features (fit on train only) ---
scaler = StandardScaler()
train_features = scaler.fit_transform(train_df[feature_cols])
test_features = scaler.transform(test_df[feature_cols])

train_target = train_df[target_col].values
test_target = test_df[target_col].values

# --- Create sliding window sequences ---
def create_sequences(features, target, lookback):
    """Create (X, y) pairs: X = window of `lookback` timesteps, y = next log return."""
    X, y = [], []
    for i in range(lookback, len(features)):
        X.append(features[i - lookback:i])
        y.append(target[i])
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_features, train_target, LOOKBACK)
X_test, y_test = create_sequences(test_features, test_target, LOOKBACK)

print(f"\nSequence shapes:")
print(f"  X_train: {X_train.shape}  y_train: {y_train.shape}")
print(f"  X_test:  {X_test.shape}   y_test:  {y_test.shape}")

# --- Convert to PyTorch tensors & DataLoaders ---
X_train_t = torch.FloatTensor(X_train).to(device)
y_train_t = torch.FloatTensor(y_train).unsqueeze(1).to(device)
X_test_t = torch.FloatTensor(X_test).to(device)
y_test_t = torch.FloatTensor(y_test).unsqueeze(1).to(device)

train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Store test dates and prices (aligned with sequences) for backtesting
test_dates = test_df.index[LOOKBACK:]
test_prices = test_df['Close'].values[LOOKBACK:]
print(f"\nTest period for backtesting: {test_dates[0].date()} to {test_dates[-1].date()} ({len(test_dates)} days)")

## 3. LSTM Model Architecture

In [ ]:
# ============================================================
# Cell 4: LSTM Model Definition
# ============================================================

class BitcoinLSTM(nn.Module):
    """LSTM model for predicting Bitcoin log returns."""
    
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super(BitcoinLSTM, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        # LSTM forward pass (default zero-initialized hidden states)
        lstm_out, _ = self.lstm(x)
        # Take only the last timestep output
        last_out = lstm_out[:, -1, :]
        # Linear projection to single prediction
        return self.fc(last_out)

# Instantiate model
num_features = len(feature_cols)
model = BitcoinLSTM(
    input_size=num_features,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
).to(device)

# Print model summary
total_params = sum(p.numel() for p in model.parameters())
print(f"Model architecture:\n{model}")
print(f"\nTotal parameters: {total_params:,}")

## 4. Training Loop

In [ ]:
# ============================================================
# Cell 5: Training Loop
# ============================================================

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=20, verbose=False
)

train_losses = []

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    n_batches = 0
    
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_loss += loss.item()
        n_batches += 1
    
    avg_loss = epoch_loss / n_batches
    train_losses.append(avg_loss)
    scheduler.step(avg_loss)
    
    # Print progress every 50 epochs
    if (epoch + 1) % 50 == 0 or epoch == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | Loss: {avg_loss:.6f} | LR: {current_lr:.6f}")

print(f"\nTraining complete. Final loss: {train_losses[-1]:.6f}")

## 5. Inference, Signal Generation & Backtesting

In [ ]:
# ============================================================
# Cell 6: Inference & Signal Generation
# ============================================================

# --- Run inference on test set ---
model.eval()
with torch.no_grad():
    predicted_log_returns = model(X_test_t).cpu().numpy().flatten()

actual_log_returns = y_test

# --- Generate threshold-based signals ---
signals = []
for pred in predicted_log_returns:
    if pred > THRESHOLD:
        signals.append('BUY')
    elif pred < -THRESHOLD:
        signals.append('SELL')
    else:
        signals.append('HOLD')

# --- Compute streak-based position sizing ---
def compute_streak(signals_list, idx):
    """Count consecutive identical signals ending at idx (max 5)."""
    if idx == 0:
        return 1
    current = signals_list[idx]
    streak = 1
    for j in range(idx - 1, max(idx - 5, -1), -1):
        if signals_list[j] == current:
            streak += 1
        else:
            break
    return min(streak, 5)

position_fractions = []
for i in range(len(signals)):
    streak = compute_streak(signals, i)
    fraction = min(streak * 0.2, 1.0)
    position_fractions.append(fraction)

print(f"Signal distribution:")
print(f"  BUY:  {signals.count('BUY')}")
print(f"  SELL: {signals.count('SELL')}")
print(f"  HOLD: {signals.count('HOLD')}")

In [ ]:
# ============================================================
# Cell 7: Backtesting Engine
# ============================================================

# --- Strategy Backtesting ---
cash = INITIAL_CAPITAL
btc_held = 0.0
portfolio_values = []
trade_log = []  # Track individual trades for win rate / profit factor

for t in range(len(test_prices)):
    price = test_prices[t]
    signal = signals[t]
    fraction = position_fractions[t]
    
    if signal == 'BUY' and cash > 0:
        amount_to_spend = cash * fraction
        btc_bought = (amount_to_spend * (1 - TX_FEE)) / price
        btc_held += btc_bought
        cash -= amount_to_spend
        trade_log.append({'type': 'BUY', 'price': price, 'amount_usd': amount_to_spend,
                          'btc': btc_bought, 'date': test_dates[t]})
    
    elif signal == 'SELL' and btc_held > 0:
        btc_to_sell = btc_held * fraction
        cash_received = (btc_to_sell * price) * (1 - TX_FEE)
        cash += cash_received
        btc_held -= btc_to_sell
        trade_log.append({'type': 'SELL', 'price': price, 'amount_usd': cash_received,
                          'btc': btc_to_sell, 'date': test_dates[t]})
    
    portfolio_values.append(cash + btc_held * price)

# Liquidate remaining BTC at final price
if btc_held > 0:
    final_price = test_prices[-1]
    liquidation_cash = (btc_held * final_price) * (1 - TX_FEE)
    cash += liquidation_cash
    btc_held = 0.0
    
final_portfolio_value = cash

# --- Buy & Hold Baseline ---
bh_btc = (INITIAL_CAPITAL * (1 - TX_FEE)) / test_prices[0]
bh_final = (bh_btc * test_prices[-1]) * (1 - TX_FEE)

# Buy & hold portfolio values over time (for plotting)
bh_portfolio_values = [bh_btc * p for p in test_prices]

portfolio_values = np.array(portfolio_values)
bh_portfolio_values = np.array(bh_portfolio_values)

print(f"Strategy final value:  ${final_portfolio_value:,.2f}")
print(f"Buy & Hold final value: ${bh_final:,.2f}")

## 6. Evaluation Metrics

In [ ]:
# ============================================================
# Cell 8: Compute All Evaluation Metrics
# ============================================================

# --- Returns ---
strategy_return = (final_portfolio_value - INITIAL_CAPITAL) / INITIAL_CAPITAL * 100
bh_return = (bh_final - INITIAL_CAPITAL) / INITIAL_CAPITAL * 100
excess_return = strategy_return - bh_return

# --- Total Trades ---
total_trades = sum(1 for t in trade_log if t['type'] in ('BUY', 'SELL'))

# --- Win Rate & Profit Factor ---
# Match BUY/SELL pairs to compute round-trip trades
buys_stack = []
gross_profits = 0.0
gross_losses = 0.0
wins = 0
completed_trades = 0

for trade in trade_log:
    if trade['type'] == 'BUY':
        buys_stack.append(trade)
    elif trade['type'] == 'SELL' and buys_stack:
        # Use weighted average buy price from accumulated buys
        avg_buy_price = np.mean([b['price'] for b in buys_stack])
        sell_price = trade['price']
        pnl = (sell_price - avg_buy_price) / avg_buy_price
        if pnl > 0:
            gross_profits += trade['amount_usd'] * pnl
            wins += 1
        else:
            gross_losses += abs(trade['amount_usd'] * pnl)
        completed_trades += 1
        buys_stack = []  # Reset after completing a round trip

win_rate = (wins / completed_trades * 100) if completed_trades > 0 else 0.0
profit_factor = (gross_profits / gross_losses) if gross_losses > 0 else float('inf')

# --- Max Drawdown ---
running_max = np.maximum.accumulate(portfolio_values)
drawdown = (portfolio_values - running_max) / running_max * 100
max_drawdown = drawdown.min()

# --- Sharpe Ratio (annualized) ---
daily_returns = np.diff(portfolio_values) / portfolio_values[:-1]
sharpe_ratio = (np.mean(daily_returns) / (np.std(daily_returns) + 1e-10)) * np.sqrt(252)

# --- Signal Counts ---
buy_count = signals.count('BUY')
sell_count = signals.count('SELL')
hold_count = signals.count('HOLD')

# Store metrics for display
metrics = {
    'strategy_return': strategy_return,
    'bh_return': bh_return,
    'excess_return': excess_return,
    'total_trades': total_trades,
    'win_rate': win_rate,
    'profit_factor': profit_factor,
    'max_drawdown': max_drawdown,
    'sharpe_ratio': sharpe_ratio,
    'buy_count': buy_count,
    'sell_count': sell_count,
    'hold_count': hold_count,
}

print("Metrics computed successfully.")

## 7. Visualizations

In [ ]:
# ============================================================
# Cell 9: Plot 1 — Training Loss Curve
# ============================================================

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(range(1, EPOCHS + 1), train_losses, color='#2196F3', linewidth=1.2)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Training Loss Over Epochs', fontsize=14, fontweight='bold')
ax.set_xlim(1, EPOCHS)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('results/plot1_training_loss.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# Cell 10: Plot 2 — Predicted vs Actual Log Returns
# ============================================================

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test_dates, actual_log_returns, color='#333333', alpha=0.6, linewidth=0.8, label='Actual Log Returns')
ax.plot(test_dates, predicted_log_returns, color='#E91E63', alpha=0.8, linewidth=0.8, label='Predicted Log Returns')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.axhline(y=THRESHOLD, color='green', linestyle=':', alpha=0.4, label=f'Threshold (±{THRESHOLD})')
ax.axhline(y=-THRESHOLD, color='red', linestyle=':', alpha=0.4)
ax.set_xlabel('Date')
ax.set_ylabel('Log Return')
ax.set_title('Predicted vs Actual Log Returns (Test Set)', fontsize=14, fontweight='bold')
ax.legend(loc='upper right')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=45)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('results/plot2_predicted_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# Cell 11: Plot 3 — Trading Signals on Price
# ============================================================

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(test_dates, test_prices, color='#333333', linewidth=1.0, label='BTC Close Price')

# Overlay BUY signals (green up triangles)
buy_mask = np.array([s == 'BUY' for s in signals])
sell_mask = np.array([s == 'SELL' for s in signals])

ax.scatter(test_dates[buy_mask], test_prices[buy_mask],
           marker='^', color='#4CAF50', s=50, alpha=0.7, label='BUY', zorder=5)
ax.scatter(test_dates[sell_mask], test_prices[sell_mask],
           marker='v', color='#F44336', s=50, alpha=0.7, label='SELL', zorder=5)

ax.set_xlabel('Date')
ax.set_ylabel('Price (USD)')
ax.set_title('Trading Signals on BTC Price', fontsize=14, fontweight='bold')
ax.legend(loc='upper left')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=45)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('results/plot3_trading_signals.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# Cell 12: Plot 4 — Strategy vs Buy & Hold Equity Curve
# ============================================================

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(test_dates, portfolio_values, color='#2196F3', linewidth=1.5, label='LSTM Strategy')
ax.plot(test_dates, bh_portfolio_values, color='#FF9800', linewidth=1.5, label='Buy & Hold', linestyle='--')

# Shade green where strategy beats B&H, red where it doesn't
ax.fill_between(test_dates, portfolio_values, bh_portfolio_values,
                where=(portfolio_values >= bh_portfolio_values),
                color='#4CAF50', alpha=0.15, label='Strategy Outperforms')
ax.fill_between(test_dates, portfolio_values, bh_portfolio_values,
                where=(portfolio_values < bh_portfolio_values),
                color='#F44336', alpha=0.15, label='Strategy Underperforms')

ax.axhline(y=INITIAL_CAPITAL, color='gray', linestyle=':', alpha=0.5, label='Initial Capital')
ax.set_xlabel('Date')
ax.set_ylabel('Portfolio Value (USD)')
ax.set_title('Strategy vs Buy & Hold', fontsize=14, fontweight='bold')
ax.legend(loc='upper left')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=45)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('results/plot4_equity_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# Cell 13: Plot 5 — Drawdown Chart
# ============================================================

fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(test_dates, drawdown, 0, color='#F44336', alpha=0.4)
ax.plot(test_dates, drawdown, color='#D32F2F', linewidth=0.8)
ax.set_xlabel('Date')
ax.set_ylabel('Drawdown (%)')
ax.set_title('Portfolio Drawdown', fontsize=14, fontweight='bold')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=45)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('results/plot5_drawdown.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# Cell 14: Plot 6 — Signal Distribution
# ============================================================

fig, ax = plt.subplots(figsize=(8, 5))
signal_labels = ['BUY', 'SELL', 'HOLD']
signal_counts = [buy_count, sell_count, hold_count]
colors = ['#4CAF50', '#F44336', '#9E9E9E']

bars = ax.bar(signal_labels, signal_counts, color=colors, edgecolor='white', linewidth=1.2)
for bar, count in zip(bars, signal_counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
            str(count), ha='center', va='bottom', fontweight='bold', fontsize=12)

ax.set_xlabel('Signal Type')
ax.set_ylabel('Count')
ax.set_title('Signal Distribution', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('results/plot6_signal_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Final Results & Output

In [ ]:
# ============================================================
# Cell 15: Print Results Summary
# ============================================================

print("=" * 44)
print("       TRADING STRATEGY RESULTS")
print("=" * 44)
print(f"Initial Capital:        ${INITIAL_CAPITAL:,.2f}")
print(f"Final Portfolio Value:  ${final_portfolio_value:,.2f}")
print(f"Strategy Return:        {strategy_return:.2f}%")
print(f"Buy & Hold Return:      {bh_return:.2f}%")
print(f"Excess Return:          {excess_return:.2f}%")
print("-" * 44)
print(f"Total Trades:           {total_trades}")
print(f"Win Rate:               {win_rate:.2f}%")
print(f"Profit Factor:          {profit_factor:.2f}")
print(f"Max Drawdown:           {max_drawdown:.2f}%")
print(f"Sharpe Ratio:           {sharpe_ratio:.2f}")
print("-" * 44)
print(f"Signal Counts:")
print(f"  BUY:  {buy_count}")
print(f"  SELL: {sell_count}")
print(f"  HOLD: {hold_count}")
print("=" * 44)

In [ ]:
# ============================================================
# Cell 16: Save Model and Results CSV
# ============================================================

# --- Save trained model ---
torch.save(model.state_dict(), 'results/lstm_model.pth')
print("Model saved to results/lstm_model.pth")

# --- Save test results CSV ---
results_df = pd.DataFrame({
    'Date': test_dates,
    'Actual_Price': test_prices,
    'Predicted_LogReturn': predicted_log_returns,
    'Signal': signals,
    'Position_Size': position_fractions,
    'Portfolio_Value': portfolio_values,
})
results_df.to_csv('results/test_results.csv', index=False)
print(f"Test results saved to results/test_results.csv ({len(results_df)} rows)")
print("\nAll outputs saved to results/ directory.")
print(f"  - results/lstm_model.pth")
print(f"  - results/test_results.csv")
print(f"  - results/plot1_training_loss.png")
print(f"  - results/plot2_predicted_vs_actual.png")
print(f"  - results/plot3_trading_signals.png")
print(f"  - results/plot4_equity_curve.png")
print(f"  - results/plot5_drawdown.png")
print(f"  - results/plot6_signal_distribution.png")